# Analyse YouTube/Netflix — EDA Demonstration

> **Auteur :** Emmanuel TSAGUE — Data Scientist / Data Analyst  
> **Données :** Simulées / Anonymisées — Portfolio pédagogique  
> **GitHub :** https://github.com/TSAGUE25


## 1. Imports

Librairies pour l'analyse et la visualisation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Imports OK')

## 2. Generation du dataset simule — 50 000 videos

Creation d'un dataset simule de videos avec metadonnees.


In [ ]:
N = 50000
DATE_MIN = datetime(2022, 1, 1)
df = pd.DataFrame({
    'categorie':   np.random.choice(['Tech','Gaming','Education','Musique','Sport','Cuisine'], N,
                       p=[0.20, 0.18, 0.15, 0.15, 0.12, 0.20]),
    'pays':        np.random.choice(['USA','Inde','Bresil','France','Autres'], N,
                       p=[0.38, 0.18, 0.11, 0.07, 0.26]),
    'vues':        np.abs(np.random.lognormal(10, 2.5, N)).astype(int),
    'duree_min':   np.abs(np.random.lognormal(2.5, 1.0, N)).clip(0.5, 120),
    'date':        [DATE_MIN + timedelta(days=np.random.randint(0, 1460)) for _ in range(N)],
})
df['date'] = pd.to_datetime(df['date'])
df['annee'] = df['date'].dt.year
df['trimestre'] = df['date'].dt.to_period('Q').astype(str)
df['engagement'] = np.random.beta(2, 20, N).clip(0.01, 0.15)
print(f'Dataset: {len(df):,} videos | Vues medianes: {df.vues.median():,.0f}')

## 3. Distribution des vues et tendances

La distribution des vues suit une loi puissance — quelques videos virales dominent.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('EDA — Donnees video simulees', fontsize=12)

axes[0].hist(np.log10(df.vues+1), bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Distribution vues (log10)')
axes[0].set_xlabel('log10(vues)')

cat_counts = df.categorie.value_counts()
axes[1].barh(cat_counts.index, cat_counts.values, color='teal')
axes[1].set_title('Videos par categorie')

trend = df.groupby('trimestre')['engagement'].mean() * 100
axes[2].plot(trend.index, trend.values, 'r-o', linewidth=2)
axes[2].set_title('Engagement moyen (%) par trimestre')
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.savefig('../figures/yt_eda_overview.png', dpi=120)
plt.show()
print('EDA complete — graphique sauvegarde')